In [ ]:
import json
import os
import sys
from pathlib import Path
from datetime import datetime
import shutil
import socket
import subprocess


# Find the repository root whether the notebook is opened from src/ or the root.
project_root = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "docker-compose.yaml").exists()
        and (candidate / "src").is_dir()
    ),
    Path.cwd(),
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src._shared import serwer
from src.Dopytywator import dopytywator
from src.Pismogenerator import pismogenerator
from src.Zrodlowyszukiwator.brief_to_evidence import (
    BriefToEvidencePipeline,
    LLMQueryPlanner,
    evidence_pack_to_markdown,
)
from src.Zrodlowyszukiwator.rag_ingestion.embedder import OllamaEmbedder
from src.Zrodlowyszukiwator.rag_retrieval import (
    HybridRetriever,
    LLMReranker,
    RetrievalFilters,
    RetrievalRepository,
)

PROJECT_ROOT = project_root
# Jupyter Lab nie wczytuje .env sam (VS Code tak), a niżej potrzebne są DATABASE_URL i OLLAMA_URL.
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")
print("Project root:", PROJECT_ROOT)

In [ ]:
# Dopytywator settings
MODEL_NAME_DOPYTYWATOR = "pismak-bielik:latest"
MAX_INTERVIEW_QUESTIONS = 1
TEMPERATURE_DOPYTYWATOR = 0.1

# Źródłowyszukiwator settings
MODEL_NAME_ZRODLOWYSZUKIWATOR = "pismak-pllum:latest"
TEMPERATURE_ZRODLOWYSZUKIWATOR = 0.1

# Pismogenerator settings
MODEL_NAME_PISMOGENERATOR = "pismak-pllum:latest"
TEMPERATURE_PISMOGENERATOR = 0.1

# Each run gets a separate output directory; generated runs are ignored by Git.

CASE_NAME = "atak_kosmitow"
OUTPUTS_DIR = PROJECT_ROOT / "outputs" / f'{datetime.now().strftime("%Y%m%d")}_{CASE_NAME}_notebook'
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
DEBUG = False

In [ ]:
# start serwer
serwer.check_available_model()
client = serwer.set_up_client(type = "local")

In [ ]:
# Sprawdź Docker Engine i dostępność PostgreSQL na porcie 5432.
DOCKER_CLI = shutil.which("docker")
if DOCKER_CLI is None:
    print("Nie znaleziono polecenia docker. Uruchom Docker Desktop albo zainstaluj Docker Desktop i otwórz ponownie VS Code.")
else:
    try:
        docker_info = subprocess.run(
            [DOCKER_CLI, "info"],
            capture_output=True,
            text=True,
            timeout=10,
        )
    except subprocess.TimeoutExpired:
        docker_info = None
        print("Docker Engine nie odpowiedział w ciągu 10 sekund. Uruchom lub poczekaj na Docker Desktop.")

    if docker_info is not None:
        if docker_info.returncode == 0:
            print("Docker Engine: działa")
        else:
            print("Docker CLI jest dostępne, ale Docker Engine nie działa. Otwórz Docker Desktop.")
            print(docker_info.stderr.strip())

try:
    with socket.create_connection(("127.0.0.1", 5432), timeout=2):
        print("PostgreSQL: port 5432 jest dostępny")
except OSError:
    print("PostgreSQL: brak połączenia na porcie 5432. Po uruchomieniu Docker Engine uruchom usługę w następnej komórce.")

# Dopytywator

In [ ]:
llm_dopytywator = serwer.LLMClient(client=client, model_name=MODEL_NAME_DOPYTYWATOR)
print("LLMClient gotowy:", MODEL_NAME_DOPYTYWATOR)

In [ ]:
# określ problem
initial_problem = dopytywator.get_problem()
initial_problem

In [ ]:
conversation = dopytywator.ask_questions(
    initial_problem = initial_problem, 
    llm = llm_dopytywator,
    max_interview_questions = MAX_INTERVIEW_QUESTIONS, 
    temperature = TEMPERATURE_DOPYTYWATOR
)

In [ ]:
brief = dopytywator.build_brief(
    messages=conversation,
    llm = llm_dopytywator
)

In [ ]:
if brief is None:
    print("Brak CaseBrief — wywiad został anulowany lub nie uruchomiony.")
else:
    print("Typ:", type(brief))
    display(brief)

In [ ]:
# 9. JSON CaseBrief — gotowy do przekazania kolejnemu etapowi
if brief is not None:
    print(brief.model_dump_json(indent=2, exclude_none=False))

# Źródłowyszukiwator

In [ ]:
# Build the CaseBrief -> EvidencePack pipeline from the existing RAG modules.
embedder = OllamaEmbedder(
    model=os.getenv("RAG_EMBEDDING_MODEL", "bge-m3"),
    base_url=os.getenv("OLLAMA_URL", "http://127.0.0.1:11434"),
)

planner = LLMQueryPlanner(
    client=client,
    model=MODEL_NAME_ZRODLOWYSZUKIWATOR,
)
retriever = HybridRetriever(
    repository=RetrievalRepository(os.environ["DATABASE_URL"]),
    embedder=embedder,
    vector_limit=20,
    lexical_limit=20,
)
reranker = LLMReranker(
    client=client,
    model=MODEL_NAME_ZRODLOWYSZUKIWATOR,
    max_selected=6,
)

zrodlowyszukiwator = BriefToEvidencePipeline(
    planner=planner,
    retriever=retriever,
    reranker=reranker,
    candidate_k=20,
    evidence_k=6,
    max_per_document=3,
)

print("Źródłowyszukiwator gotowy")

In [ ]:
# Run retrieval for the current brief.
evidence_pack = zrodlowyszukiwator.run(
    brief,
    filters=RetrievalFilters(city="Wrocław"),
)
context_markdown = evidence_pack_to_markdown(evidence_pack)
print(context_markdown)

In [ ]:
# Zapis evidence.json obok brief.json.
evidence_path = OUTPUTS_DIR / "evidence.json"
evidence_path.write_text(
    evidence_pack.model_dump_json(indent=2),
    encoding="utf-8",
)

print("Zapisano:", evidence_path)

# Pismogenerator

In [ ]:
llm_pismogenerator = serwer.LLMClient(client=client, model_name=MODEL_NAME_PISMOGENERATOR)
print("LLMClient gotowy:", MODEL_NAME_PISMOGENERATOR)

In [ ]:
draft = pismogenerator.build_draft(
    case_brief=brief,
    evidence_pack=evidence_pack,
    llm=llm_pismogenerator,
    llm_temperature=TEMPERATURE_PISMOGENERATOR,
)

In [ ]:
letter = pismogenerator.build_letter(
    draft=draft,
    evidence_pack=evidence_pack,
    user_evidence=brief.available_evidence,
)

# save

In [ ]:
brief_path = OUTPUTS_DIR / "brief.json"
conversation_path = OUTPUTS_DIR / "conversation.json"
draft_path = OUTPUTS_DIR / "draft.json"
letter_path = OUTPUTS_DIR / "letter.md"

brief_path.write_text(
    brief.model_dump_json(indent=2, exclude_none=False),
    encoding="utf-8",
)
conversation_path.write_text(
    json.dumps([m.model_dump(mode="json") for m in conversation], ensure_ascii=False, indent=2),
    encoding="utf-8",
)
draft_path.write_text(
    draft.model_dump_json(
        indent=2,
        exclude_none=False,
    ),
    encoding="utf-8",
)
letter_path.write_text(
    letter,
    encoding="utf-8",
)


print("Zapisano:")
print(" -", brief_path.resolve())
print(" -", conversation_path.resolve())
print(" -", draft_path.resolve())
print(" -", letter_path.resolve())
